In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

warnings.filterwarnings("ignore")

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"
MODELS_DIR = PROJECT_ROOT / "MODELS"

FEATURE_FILE = PROCESSED_DIR / "cars_features.csv"

In [3]:
cars = pd.read_csv(FEATURE_FILE)

print("Dataset shape:", cars.shape)

cars.head()

Dataset shape: (1725, 10)


,id,year,brand,full_model_name,model_name,price,distance_travelled(kms),fuel_type,city,car_age
0,0,2016,Honda,Honda Brio S MT,Brio,425000.0,9680.0,Petrol,Mumbai,10
1,1,2012,Nissan,Nissan Sunny XV Diesel,Sunny,325000.0,119120.0,Diesel,Mumbai,14
2,2,2017,Toyota,Toyota Fortuner 2.8 4x2 MT [2016-2020],Fortuner,2650000.0,64593.0,Diesel,Thane,9
3,3,2017,Mercedes-Benz,Mercedes-Benz E-Class E 220d Expression [2019-...,E-Class,4195000.0,25000.0,Diesel,Mumbai,9
4,4,2012,Hyundai,Hyundai Verna Fluidic 1.6 CRDi SX,Verna,475000.0,23800.0,Diesel,Mumbai,14


In [4]:
price_candidates = [
    "selling_price",
    "price",
    "resale_price"
]

TARGET = next(
    (
        col for col in price_candidates
        if col in cars.columns
    ),
    None
)

if TARGET is None:
    raise ValueError("Target price column not found.")

print("Target:", TARGET)

Target: price


In [5]:
X = cars.drop(columns=[TARGET])
y = cars[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1725, 9)
y shape: (1725,)


In [6]:
# Columns that should not directly determine the prediction
LEAKAGE_COLUMNS = [
    "log_price"
]

LEAKAGE_COLUMNS = [
    col for col in LEAKAGE_COLUMNS
    if col in X.columns
]

X = X.drop(columns=LEAKAGE_COLUMNS)

print("Removed:", LEAKAGE_COLUMNS)

Removed: []


In [7]:
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['id', 'year', 'distance_travelled(kms)', 'car_age']

Categorical features:
['brand', 'full_model_name', 'model_name', 'fuel_type', 'city']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 1380
Testing rows: 345


In [9]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [10]:
models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

print("Models:")
for name in models:
    print("-", name)

Models:
- Linear Regression
- Random Forest
- Gradient Boosting


In [11]:
trained_models = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(
        X_train,
        y_train
    )

    trained_models[name] = pipeline

    print(f"{name} training complete.")


Training Linear Regression...
Linear Regression training complete.

Training Random Forest...
Random Forest training complete.

Training Gradient Boosting...
Gradient Boosting training complete.


In [12]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

training_results = []

for name, model in trained_models.items():

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    training_results.append({
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(training_results)

results_df.sort_values(
    "RMSE"
)

,model,MAE,RMSE,R2
1,Random Forest,329445.458937,9.898650e+05,0.708124
2,Gradient Boosting,495968.426892,1.076627e+06,0.654715
0,Linear Regression,571589.712184,1.094631e+06,0.643071


In [13]:
for name, model in trained_models.items():

    filename = (
        name.lower()
        .replace(" ", "_")
        .replace("-", "")
        + ".joblib"
    )

    filepath = MODELS_DIR / filename

    joblib.dump(
        model,
        filepath
    )

    print("Saved:", filepath)

Saved: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\MODELS\linear_regression.joblib
Saved: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\MODELS\random_forest.joblib
Saved: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\MODELS\gradient_boosting.joblib


In [14]:
best_model_name = (
    results_df
    .sort_values("RMSE")
    .iloc[0]["model"]
)

best_model = trained_models[
    best_model_name
]

BEST_MODEL_PATH = (
    MODELS_DIR /
    "car_price_model.joblib"
)

joblib.dump(
    best_model,
    BEST_MODEL_PATH
)

print("Best model:", best_model_name)
print("Saved to:", BEST_MODEL_PATH)

Best model: Random Forest
Saved to: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\MODELS\car_price_model.joblib
